# Анализ образовательных метрик

Добро пожаловать в **EdTech**!

Продакт-менеджер Василий попросил проанализировать поведение студентов на курсах, предметах и экзаменах онлайн-школы и ответить на следующие вопросы.

## Задания

### 1. Успешная сдача курсов

Сколько студентов успешно сдали **только один курс**?

> Успешная сдача курса — получение зачёта по курсу на экзамене.

---

### 2. Завершаемость экзаменов

Найдите и отсортируйте `id` экзаменов в рамках курса по возрастанию уровня завершаемости.

Завершаемость рассчитывается по формуле:

[
\text{Завершаемость} =
\frac{\text{Количество успешно сданных экзаменов}}
{\text{Количество всех попыток сдать экзамен}}
\times 100%
]

---

### 3. Самые популярные предметы

Выявите **ТОП-3 самых популярных предмета** по количеству регистраций студентов.

---

### 4. Предметы с наибольшим оттоком

Выявите **ТОП-3 предмета с самым большим оттоком студентов**.

---

### 5. Семестр с самой низкой завершаемостью

Используя `pandas`, в период с начала **2013 года** по конец **2014 года** определите семестр с самой низкой завершаемостью курсов.

---

### 6. Семестр с самыми долгими сроками сдачи

Используя `pandas`, в период с начала **2013 года** по конец **2014 года** определите семестр с самыми долгими средними сроками сдачи курсов.

---

## 7. RFM-сегментация студентов

Для качественного анализа аудитории часто используются подходы, основанные на сегментации.

Используя Python, постройте адаптированные **RFM-кластеры студентов**, чтобы оценить аудиторию онлайн-школы.

В адаптированной RFM-кластеризации используйте следующие метрики:

* **R — Recency:** среднее время сдачи одного экзамена;
* **F — Frequency:** завершаемость курсов;
* **M — Monetary:** среднее количество баллов, получаемое студентом за экзамен.

Для каждого RFM-сегмента определите границы метрик:

* `recency`;
* `frequency`;
* `monetary`.

Это необходимо для последующей интерпретации полученных кластеров.

### Вопросы

#### 7.1

Чему равна минимальная граница по `recency`?

#### 7.2

Чему равна максимальная граница по `recency`?

#### 7.3

Чему равна минимальная граница по `monetary`?

#### 7.4

Чему равна максимальная граница по `monetary`?

#### 7.5

Сколько студентов попадут в кластер `232`?

---

## Правила формирования RFM-кластеров

### Recency

* студент получает `2`, если значение `recency` меньше или равно медиане по `recency`;
* студент получает `1`, если значение `recency` больше медианы.

```python
if recency <= recency_median:
    r = 2
else:
    r = 1
```

### Frequency

* студент получает `1`, если `frequency < 50`;
* студент получает `2`, если `50 <= frequency < 100`;
* студент получает `3`, если `frequency >= 100`.

```python
if frequency < 50:
    f = 1
elif frequency < 100:
    f = 2
else:
    f = 3
```

### Monetary

* студент получает `1`, если `monetary < 40`;
* студент получает `2`, если `40 <= monetary <= 80`;
* студент получает `3`, если `monetary > 80`.

```python
if monetary < 40:
    m = 1
elif monetary <= 80:
    m = 2
else:
    m = 3
```

Итоговый RFM-кластер формируется объединением значений `R`, `F` и `M`.

Например:

```python
rfm_segment = str(r) + str(f) + str(m)
```

Студент из сегмента `232` имеет следующие характеристики:

* `R = 2` — сдаёт экзамены быстрее или на уровне медианного значения;
* `F = 3` — завершаемость курсов равна или превышает 100%;
* `M = 2` — средний балл находится в диапазоне от 40 до 80 включительно.

---

## Предварительное исследование данных

Перед решением задач необходимо провести предварительный анализ данных и сформулировать, что именно должно считаться **курсом**.

Выбор определения курса необходимо обосновать с помощью:

1. фактов сдачи экзаменов;
2. распределения студентов;
3. уникального идентификатора курса;
4. связи предмета, семестра и экзамена.

Возможный вариант определения:

> Курс — это комбинация идентификатора предмета и семестра его проведения.

Например:

```python
df["course_id"] = (
    df["code_module"].astype(str)
    + "_"
    + df["code_presentation"].astype(str)
)
```

Такой подход позволяет разделить один и тот же предмет, проводившийся в разные семестры, на отдельные курсы.


# Описание данных

## 1. `Assessments.csv`

Файл содержит информацию о тестах и экзаменах.

Обычно каждый предмет в рамках семестра включает несколько тестов с оценками, после которых проводится итоговый экзамен.

| Поле                | Описание                                                                                                             |
| ------------------- | -------------------------------------------------------------------------------------------------------------------- |
| `code_module`       | Идентификационный код предмета.                                                                                      |
| `code_presentation` | Идентификационный код семестра.                                                                                      |
| `id_assessment`     | Уникальный идентификационный номер теста или экзамена.                                                               |
| `assessment_type`   | Тип оценивания.                                                                                                      |
| `date`              | Крайний срок сдачи теста, выраженный в количестве дней с начала семестра. Первый день семестра обозначается как `0`. |
| `weight`            | Вес теста в итоговой оценке за курс, выраженный в процентах.                                                         |

### Типы оценивания

Поле `assessment_type` может принимать следующие значения:

* `TMA` — оценивание преподавателем;
* `CMA` — компьютерное оценивание;
* `Exam` — итоговый экзамен по курсу.

Обычно итоговый экзамен рассматривается отдельно и имеет вес `100%`.

Суммарный вес всех остальных тестов в рамках курса также обычно составляет `100%`.

---

## 2. `Courses.csv`

Файл содержит список предметов, проводимых в разных семестрах.

| Поле                         | Описание                           |
| ---------------------------- | ---------------------------------- |
| `code_module`                | Идентификационный код предмета.    |
| `code_presentation`          | Идентификационный код семестра.    |
| `module_presentation_length` | Продолжительность семестра в днях. |

Комбинацию полей `code_module` и `code_presentation` можно рассматривать как уникальный идентификатор запуска курса:

```python
courses["course_id"] = (
    courses["code_module"].astype(str)
    + "_"
    + courses["code_presentation"].astype(str)
)
```

Например, предмет `AAA`, проводившийся в семестре `2013J`, можно обозначить как курс:

```text
AAA_2013J
```

---

## 3. `StudentAssessment.csv`

Файл содержит результаты сдачи тестов студентами.

Если студент не отправил работу на проверку, информация о ней отсутствует в таблице.

| Поле             | Описание                                                                    |
| ---------------- | --------------------------------------------------------------------------- |
| `id_assessment`  | Идентификационный номер теста или экзамена.                                 |
| `id_student`     | Уникальный идентификатор студента.                                          |
| `date_submitted` | Дата сдачи теста студентом, выраженная в количестве дней с начала семестра. |
| `is_banked`      | Признак перезачёта результата из предыдущего семестра.                      |
| `score`          | Оценка студента за тест или экзамен.                                        |

### Интерпретация поля `is_banked`

Поле `is_banked` показывает, был ли результат теста перезачтён из предыдущего семестра.

Такое возможно, например, если студент вернулся после академического отпуска.

Возможные значения:

* `0` — тест был сдан в текущем семестре;
* `1` — результат был перезачтён из прошлого семестра.

### Интерпретация поля `score`

Оценка находится в диапазоне от `0` до `100`.

* `score < 40` — неуспешная сдача;
* `score >= 40` — успешная сдача.

Пример создания признака успешной сдачи:

```python
student_assessment["is_passed"] = (
    student_assessment["score"] >= 40
)
```

---

## 4. `StudentRegistration.csv`

Файл содержит информацию о регистрации студентов на предметы в рамках конкретного семестра.

| Поле                  | Описание                                                                             |
| --------------------- | ------------------------------------------------------------------------------------ |
| `code_module`         | Идентификационный код предмета.                                                      |
| `code_presentation`   | Идентификационный код семестра.                                                      |
| `id_student`          | Уникальный идентификатор студента.                                                   |
| `date_registration`   | Дата регистрации на курс, выраженная в количестве дней относительно начала семестра. |
| `date_unregistration` | Дата отмены регистрации на курс, выраженная в количестве дней с начала семестра.     |

### Интерпретация поля `date_registration`

Значение показывает, за сколько дней до или после начала семестра студент зарегистрировался на курс.

Например:

```text
date_registration = -30
```

означает, что студент зарегистрировался за `30` дней до начала семестра.

Положительное значение означает, что регистрация произошла уже после начала семестра.

### Интерпретация поля `date_unregistration`

Поле содержит дату отмены регистрации студентом.

* если значение указано, студент отменил регистрацию или покинул курс;
* если значение отсутствует, студент не отменял регистрацию и считается оставшимся на курсе до его окончания.

Пример создания признака оттока:

```python
student_registration["is_churned"] = (
    student_registration["date_unregistration"].notna()
)
```

---

# Связи между таблицами

Основные связи между файлами можно представить следующим образом:

```text
Courses
    |
    | code_module + code_presentation
    |
Assessments
    |
    | id_assessment
    |
StudentAssessment
```

Таблица `StudentRegistration` связывается с курсами по следующим полям:

```text
code_module + code_presentation
```

Для анализа конкретного студента дополнительно используется:

```text
id_student
```

Таким образом:

* `Courses.csv` описывает запуски курсов;
* `Assessments.csv` описывает тесты и экзамены внутри курсов;
* `StudentAssessment.csv` содержит результаты студентов;
* `StudentRegistration.csv` содержит регистрации и отмены регистраций.


## Кейс 1
Сколько студентов успешно сдали только один курс?

* для подсчёта числа успешно сданных экзаменов нужно использовать id_assessment



In [1]:
import pandas as pd

In [2]:
Assessments = pd.read_csv('Assessments.csv')
StudentAssessment = pd.read_csv('StudentAssessment.csv')

In [3]:
Assessments.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


In [4]:
StudentAssessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [5]:
merging = pd.merge(Assessments, StudentAssessment, on='id_assessment')
merging.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
0,AAA,2013J,1752,TMA,19.0,10.0,11391,18,0,78.0
1,AAA,2013J,1752,TMA,19.0,10.0,28400,22,0,70.0
2,AAA,2013J,1752,TMA,19.0,10.0,31604,17,0,72.0
3,AAA,2013J,1752,TMA,19.0,10.0,32885,26,0,69.0
4,AAA,2013J,1752,TMA,19.0,10.0,38053,19,0,79.0


In [8]:
success_exams = merging.query('assessment_type == "Exam" and score >= 40 ')
success_exams.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
52924,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0
52925,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0
52926,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0
52927,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0
52928,CCC,2014B,24290,Exam,NaN,100.0,560494,230,0,92.0


In [14]:
duplicates = success_exams.groupby(['id_student', 'id_assessment']).size()
duplicates[duplicates > 1].head(10)

Series([], dtype: int64)

In [15]:
success_unique = success_exams[['id_student', 'id_assessment', 
                                 'code_module', 'code_presentation']].drop_duplicates()

print(f'Уникальных записей студент-экзамен: {len(success_unique)}')

Уникальных записей студент-экзамен: 4392


In [16]:
success_unique['course'] = success_unique['code_module'] + '_' + success_unique['code_presentation']

courses_per_student = success_unique.groupby('id_student')['course'].nunique()

print(courses_per_student.value_counts().sort_index().head(10))

course
1    3802
2     295
Name: count, dtype: int64


In [17]:
students_with_one = courses_per_student[courses_per_student == 1]
answer = len(students_with_one)

print(f'Студентов, сдавших только один курс: {answer}')

Студентов, сдавших только один курс: 3802


## Кейс 2
Найдите и отсортируйте id экзаменов в рамках курса по возрастанию уровня завершаемости (1 - самый маленький, 6 - самый большой уровень завершаемости):



In [25]:
all_exams = merging.query('assessment_type == "Exam"')

In [26]:
all_exams.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
52923,CCC,2014B,24290,Exam,NaN,100.0,558914,230,0,32.0
52924,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0
52925,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0
52926,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0
52927,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0


In [29]:
completion = all_exams.groupby('id_assessment', as_index=False) \
                      .agg(total_attempts = ( 'score' , 'count'), successful = ( 'score', lambda x: (x>=40).sum()) )

In [31]:
completion['completion_rate'] = completion.successful / completion.total_attempts *100

In [32]:
completion.head()

,id_assessment,total_attempts,successful,completion_rate
0,24290,747,664,88.888889
1,24299,1168,1019,87.243151
2,25340,602,504,83.720930
3,25354,968,878,90.702479
4,25361,524,485,92.557252


In [35]:
completion.sort_values('completion_rate')

,id_assessment,total_attempts,successful,completion_rate
2,25340,602,504,83.720930
1,24299,1168,1019,87.243151
5,25368,950,842,88.631579
0,24290,747,664,88.888889
3,25354,968,878,90.702479
4,25361,524,485,92.557252


## Кейс 3
Выявите самые популярные предметы (ТОП-3) по количеству регистраций на них:

In [36]:
StudentRegistration = pd.read_csv('StudentRegistration.csv') 
StudentRegistration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [39]:
StudentRegistration.groupby('code_module', as_index=False).agg(top = ('id_student', 'count')).sort_values('top', ascending=False).head(3)

,code_module,top
1,BBB,7909
5,FFF,7762
3,DDD,6272


## Кейс 4
Выявите предметы с самым большим оттоком* (ТОП-3):

In [41]:
StudentRegistration.groupby('code_module', as_index=False).agg( un_top = ('date_unregistration', 'count')).sort_values('un_top', ascending=False).head(3)

,code_module,un_top
5,FFF,2380
1,BBB,2377
3,DDD,2235


## Кейс 5
Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самой низкой завершаемостью курсов:

In [42]:
all_exams.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
52923,CCC,2014B,24290,Exam,NaN,100.0,558914,230,0,32.0
52924,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0
52925,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0
52926,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0
52927,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0


In [44]:
exams_2013_2014 = all_exams[ all_exams.code_presentation.str.startswith('2013') | all_exams.code_presentation.str.startswith('2014') ]

In [45]:
exams_2013_2014.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
52923,CCC,2014B,24290,Exam,NaN,100.0,558914,230,0,32.0
52924,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0
52925,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0
52926,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0
52927,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0


In [46]:
print(exams_2013_2014['code_presentation'].unique())


['2014B' '2014J' '2013B' '2013J']


In [50]:
completion_by_semester = exams_2013_2014.groupby('code_presentation', as_index=False).agg(
    total=('score', 'count'),
    success=('score', lambda x: (x >= 40).sum())
)

completion_by_semester['completion_rate'] = (
    completion_by_semester['success'] / completion_by_semester['total'] * 100
)

In [51]:
completion_by_semester_sorted = completion_by_semester.sort_values('completion_rate')
completion_by_semester_sorted

,code_presentation,total,success,completion_rate
0,2013B,602,504,83.720930
3,2014J,2118,1861,87.865911
2,2014B,1271,1149,90.401259
1,2013J,968,878,90.702479


## Кейс 6
Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самыми долгими средними сроками сдачи курсов:

In [54]:
long_srok = exams_2013_2014.groupby('code_presentation', as_index=False).date_submitted.mean()
long_srok.sort_values('date_submitted', ascending=False)

,code_presentation,date_submitted
3,2014J,243.674221
1,2013J,239.505165
2,2014B,232.988985
0,2013B,230.179402


## Кейс 7
Часто для качественного анализа аудитории используют подходы, основанные на сегментации. Используя python, постройте адаптированные RFM-кластеры студентов, чтобы качественно оценить свою аудиторию. В адаптированной кластеризации можете выбрать следующие метрики:

R — среднее время сдачи одного экзамена,

F — завершаемость курсов,

M — среднее количество баллов, получаемое за экзамен.

Для каждого RFM-сегмента постройте границы метрик recency, frequency и monetary для интерпретации этих кластеров.


Чему равна минимальная граница по recency?



In [55]:
all_exams.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
52923,CCC,2014B,24290,Exam,NaN,100.0,558914,230,0,32.0
52924,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0
52925,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0
52926,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0
52927,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0


In [56]:
all_exams.date_submitted.min()

229

## Кейс 8
Чему равна максимальная граница по recency?

In [57]:
all_exams.date_submitted.max()

285

## Кейс 9
Чему равна минимальная граница по monetary?



In [58]:
all_exams.score.min()

0.0

## Кейс 10
Чему равна максимальная граница по monetary?

In [59]:
all_exams.score.max()

100.0

## Кейс 11
Используя логику:

1) по recency можно получить 2, если значение recency меньше либо равно медиане по recency. В остальных случаях — 1.

2) по frequency можно получить 1, если значение frequency меньше 50. Можно получить 2, если значение по frequency меньше 100. Можно получить 3 в остальных случаях.

3) по monetary можно получить 1, если значение monetary меньше 40. Можно получить 2, если значение по monetary меньше либо равно 80. Можно получить 3 в остальных случаях.

Сколько клиентов попадут в кластер 232?

In [92]:
median_recency = all_exams.date_submitted.median()

In [93]:
R = all_exams.groupby('id_student')['date_submitted'].mean().reset_index()
R.columns = ['id_student', 'R']

M = all_exams.groupby('id_student')['score'].mean().reset_index()
M.columns = ['id_student', 'M']

In [94]:
total_exams = all_exams.groupby('id_student').size().reset_index()
total_exams.columns = ['id_student', 'total_exams']

success_exams = all_exams.query('score >= 40').groupby('id_student').size().reset_index()
success_exams.columns = ['id_student', 'success_exams']

F_df = total_exams.merge(success_exams, on='id_student', how='left')
F_df['success_exams'] = F_df['success_exams'].fillna(0)
F_df['F'] = F_df['success_exams'] / F_df['total_exams'] * 100

In [95]:
students_with_exams = all_exams[['id_student']].drop_duplicates()

RFM = students_with_exams.merge(R, on='id_student', how='left')
RFM = RFM.merge(F_df[['id_student', 'F']], on='id_student', how='left')
RFM = RFM.merge(M, on='id_student', how='left')

RFM['F'] = RFM['F'].fillna(0)
RFM['M'] = RFM['M'].fillna(0)


In [96]:
median_R = RFM['R'].median()
RFM['R_score'] = np.where(RFM['R'] <= median_R, 2, 1)
RFM['F_score'] = np.where(RFM['F'] < 50, 1, 
                    np.where(RFM['F'] < 100, 2, 3))
RFM['M_score'] = np.where(RFM['M'] < 40, 1, 
                    np.where(RFM['M'] <= 80, 2, 3))


In [97]:
RFM['Cluster'] = (RFM['R_score'].astype(str) + 
                  RFM['F_score'].astype(str) + 
                  RFM['M_score'].astype(str))


In [98]:
cluster_232 = RFM[RFM['Cluster'] == '232']
len(cluster_232)

1555